# Lab 5 — Clean up

Deletes the endpoints, endpoint configs and models created by lab 4 (matched by the `openvla-lora-` name
prefix). Model packages, MLflow runs and S3 data stay unless you set the flags below.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

In [ ]:
DELETE_MODEL_PACKAGES = False   # the model package group and all its versions
DELETE_S3_DATA = False          # everything under s3_root (datasets, checkpoints, training/eval output)

prefix = f"{config.PROJECT}-"
for ep in (e for page in sm_client.get_paginator("list_endpoints").paginate(NameContains=prefix) for e in page["Endpoints"]):
    sm_client.delete_endpoint(EndpointName=ep["EndpointName"])
    print("deleted endpoint", ep["EndpointName"])
for cfg in (c for page in sm_client.get_paginator("list_endpoint_configs").paginate(NameContains=prefix) for c in page["EndpointConfigs"]):
    sm_client.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])
    print("deleted endpoint config", cfg["EndpointConfigName"])
for m in (m for page in sm_client.get_paginator("list_models").paginate(NameContains=prefix) for m in page["Models"]):
    sm_client.delete_model(ModelName=m["ModelName"])
    print("deleted model", m["ModelName"])

In [ ]:
if DELETE_MODEL_PACKAGES:
    for p in (p for page in sm_client.get_paginator("list_model_packages").paginate(ModelPackageGroupName=config.MODEL_PACKAGE_GROUP) for p in page["ModelPackageSummaryList"]):
        sm_client.delete_model_package(ModelPackageName=p["ModelPackageArn"])
    sm_client.delete_model_package_group(ModelPackageGroupName=config.MODEL_PACKAGE_GROUP)
    print("deleted", config.MODEL_PACKAGE_GROUP)

if DELETE_S3_DATA:
    bucket, prefix = s3_root[5:].split("/", 1)
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/"):
        keys = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": keys})
    print("deleted", s3_root)